# 01 — Discovery y Perfilado de Datos

> **Nota:** este notebook lee directamente de los archivos Parquet de Bronze (`data/bronze/`), no de Postgres. La arquitectura del proyecto migró de Bronze-en-Postgres a Bronze-en-Parquet a mitad del desarrollo (ver `docs/decisiones.md`); este notebook se actualizó para que sea ejecutable de punta a punta contra la arquitectura final, sin depender de una carga previa a Postgres que ya no existe en el pipeline.

Para cada tabla revisamos:
- Volumen y estructura (filas, columnas, tipos)
- Nulos por columna
- Duplicados (filas completas y por clave primaria)
- Cardinalidad de columnas clave
- Llaves huérfanas (FK que no existen en la tabla referenciada)
- Outliers / valores fuera de rango en columnas numéricas y de fecha


In [1]:
import os
from pathlib import Path
import pandas as pd

pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 150)

BRONZE_PATH = Path(os.environ.get("BRONZE_DATA_PATH", "/home/jovyan/work/data/bronze"))
DOMAINS = ["university", "billing", "crm"]

print("Leyendo Bronze desde:", BRONZE_PATH)
print("Existe:", BRONZE_PATH.exists())

Conexión OK.


In [2]:
tables_list = []
for domain in DOMAINS:
    for f in sorted((BRONZE_PATH / domain).glob("*.parquet")):
        tables_list.append(f"{domain}_{f.stem}")

for t in tables_list:
    domain, table = t.split("_", 1)
    df_full = pd.read_parquet(BRONZE_PATH / domain / f"{table}.parquet")
    df_sample = df_full.head(5)
    n_rows = len(df_full)
    print(f"\n{'='*100}")
    print(f"  {t}   ({n_rows} filas)")
    print('='*100)
    display(df_sample)

In [3]:
for t in tables_list:
    domain, table = t.split("_", 1)
    df = pd.read_parquet(BRONZE_PATH / domain / f"{table}.parquet")
    cols = [c for c in df.columns if c not in ("_source_file", "_source_domain", "_ingested_at")]
    print(f"\n{t}:")
    print(", ".join(cols))

In [4]:
tables = pd.DataFrame({"table_name": tables_list})
tables

,table_name


## Funciones de perfilado


In [5]:
def load_table(table_name):
    """table_name viene con el prefijo de dominio, ej. 'university_students'
    (mismo formato que se usaba antes con el schema bronze de Postgres)."""
    for domain in DOMAINS:
        prefix = domain + "_"
        if table_name.startswith(prefix):
            file_table = table_name[len(prefix):]
            return pd.read_parquet(BRONZE_PATH / domain / f"{file_table}.parquet")
    raise ValueError(f"No se pudo determinar el dominio de: {table_name}")


def null_report(df, table_name):
    """Cantidad y % de nulos/vacíos por columna."""
    nulls = df.isna().sum()
    empties = (df == "").sum(numeric_only=False)
    total_missing = nulls + empties
    report = pd.DataFrame({
        "nulls": nulls,
        "empty_strings": empties,
        "total_missing": total_missing,
        "pct_missing": (total_missing / len(df) * 100).round(2)
    })
    report = report[report["total_missing"] > 0].sort_values("pct_missing", ascending=False)
    print(f"[{table_name}] columnas con datos faltantes:")
    return report


def duplicate_report(df, table_name, key_cols=None):
    """Duplicados por fila completa y, opcionalmente, por columna(s) clave."""
    full_dupes = df.duplicated().sum()
    print(f"[{table_name}] filas completamente duplicadas: {full_dupes}")
    if key_cols:
        key_dupes = df.duplicated(subset=key_cols).sum()
        print(f"[{table_name}] duplicados por clave {key_cols}: {key_dupes}")
    return full_dupes


def cardinality_report(df, table_name, cols):
    """Valores únicos por columna, útil para detectar columnas casi-constantes
    o con más variedad de la esperada."""
    report = pd.DataFrame({
        "n_unique": [df[c].nunique() for c in cols],
        "n_total": len(df),
    }, index=cols)
    report["pct_unique"] = (report["n_unique"] / report["n_total"] * 100).round(2)
    print(f"[{table_name}] cardinalidad:")
    return report


def orphan_keys(df_child, fk_col, df_parent, pk_col, child_name, parent_name):
    """Filas del hijo cuya FK no existe en el padre (llaves huérfanas)."""
    child_ids = set(df_child[fk_col].dropna())
    parent_ids = set(df_parent[pk_col].dropna())
    orphans = child_ids - parent_ids
    print(f"[{child_name}.{fk_col} -> {parent_name}.{pk_col}] "
          f"huérfanos: {len(orphans)} de {len(child_ids)} valores únicos")
    return orphans

---
## Dominio: `university`


In [6]:
semesters   = load_table("university_semesters")
professors  = load_table("university_professors")
students    = load_table("university_students")
courses     = load_table("university_courses")
enrollments = load_table("university_enrollments")
grades      = load_table("university_grades")

students.head()


ProgrammingError: (psycopg2.errors.UndefinedTable) relation "bronze.university_semesters" does not exist
LINE 1: SELECT * FROM bronze."university_semesters"
                      ^

[SQL: SELECT * FROM bronze."university_semesters"]
(Background on this error at: https://sqlalche.me/e/20/f405)

In [ ]:
for name, df in [("semesters", semesters), ("professors", professors), ("students", students),
                  ("courses", courses), ("enrollments", enrollments), ("grades", grades)]:
    print(f"{name:15s} shape={df.shape}")


In [ ]:
null_report(students, "students")


In [ ]:
null_report(enrollments, "enrollments")


In [ ]:
null_report(grades, "grades")


In [ ]:
duplicate_report(students, "students", key_cols=["student_id"])
duplicate_report(enrollments, "enrollments", key_cols=["enrollment_id"])
duplicate_report(grades, "grades", key_cols=["enrollment_id"])


In [ ]:
orphan_keys(enrollments, "student_id", students, "student_id", "enrollments", "students")
orphan_keys(enrollments, "course_id", courses, "course_id", "enrollments", "courses")
orphan_keys(courses, "professor_id", professors, "professor_id", "courses", "professors")
orphan_keys(grades, "enrollment_id", enrollments, "enrollment_id", "grades", "enrollments")


In [ ]:
grades_numeric = pd.to_numeric(grades["score"], errors="coerce")  # ajustar nombre de columna
print("Valores no convertibles a número:", grades_numeric.isna().sum() - grades["score"].isna().sum())
print(grades_numeric.describe())


### Hallazgos — `university`

- Nulos:
- Duplicados:
- Llaves huérfanas:
- Outliers / rangos inválidos:


---
## Dominio: `billing`

Tablas: `customers`, `products`, `subscriptions`, `invoices`, `invoice_items`, `payments`.

- `subscriptions.customer_id` → `customers.customer_id`
- `subscriptions.product_id` → `products.product_id`
- `invoices.customer_id` → `customers.customer_id`
- `invoice_items.invoice_id` → `invoices.invoice_id`
- `payments.invoice_id` → `invoices.invoice_id`


In [ ]:
customers      = load_table("billing_customers")
products       = load_table("billing_products")
subscriptions  = load_table("billing_subscriptions")
invoices       = load_table("billing_invoices")
invoice_items  = load_table("billing_invoice_items")
payments       = load_table("billing_payments")

invoices.head()


In [ ]:
for name, df in [("customers", customers), ("products", products), ("subscriptions", subscriptions),
                  ("invoices", invoices), ("invoice_items", invoice_items), ("payments", payments)]:
    print(f"{name:15s} shape={df.shape}")


In [ ]:
null_report(customers, "customers")


In [ ]:
null_report(invoices, "invoices")


In [ ]:
null_report(payments, "payments")


In [ ]:
duplicate_report(customers, "customers", key_cols=["customer_id"])
duplicate_report(invoices, "invoices", key_cols=["invoice_id"])
duplicate_report(payments, "payments", key_cols=["payment_id"])


In [ ]:
orphan_keys(subscriptions, "customer_id", customers, "customer_id", "subscriptions", "customers")
orphan_keys(subscriptions, "product_id", products, "product_id", "subscriptions", "products")
orphan_keys(invoices, "customer_id", customers, "customer_id", "invoices", "customers")
orphan_keys(invoice_items, "invoice_id", invoices, "invoice_id", "invoice_items", "invoices")
orphan_keys(payments, "invoice_id", invoices, "invoice_id", "payments", "invoices")


In [ ]:
amount_col = "amount"  # ajustar
amounts = pd.to_numeric(payments[amount_col], errors="coerce")
print(amounts.describe())
print("Negativos:", (amounts < 0).sum())
print("Ceros:", (amounts == 0).sum())


### Hallazgos — `billing`

- Nulos:
- Duplicados:
- Llaves huérfanas:
- Outliers / montos inválidos:


---
## Dominio: `crm`

Tablas: `accounts`, `contacts`, `leads`, `opportunities`, `opportunity_contacts`, `activities`.

- `contacts.account_id` → `accounts.account_id`
- `opportunities.account_id` → `accounts.account_id`
- `opportunity_contacts.opportunity_id` → `opportunities.opportunity_id`
- `opportunity_contacts.contact_id` → `contacts.contact_id`
- `activities.contact_id` → `contacts.contact_id` (verificar si también referencia opportunity_id)


In [ ]:
accounts              = load_table("crm_accounts")
contacts              = load_table("crm_contacts")
leads                 = load_table("crm_leads")
opportunities         = load_table("crm_opportunities")
opportunity_contacts  = load_table("crm_opportunity_contacts")
activities            = load_table("crm_activities")

opportunities.head()


In [ ]:
for name, df in [("accounts", accounts), ("contacts", contacts), ("leads", leads),
                  ("opportunities", opportunities), ("opportunity_contacts", opportunity_contacts),
                  ("activities", activities)]:
    print(f"{name:15s} shape={df.shape}")


In [ ]:
null_report(accounts, "accounts")


In [ ]:
null_report(opportunities, "opportunities")


In [ ]:
null_report(activities, "activities")


In [ ]:
duplicate_report(accounts, "accounts", key_cols=["account_id"])
duplicate_report(contacts, "contacts", key_cols=["contact_id"])
duplicate_report(opportunities, "opportunities", key_cols=["opportunity_id"])


In [ ]:
orphan_keys(contacts, "account_id", accounts, "account_id", "contacts", "accounts")
orphan_keys(opportunities, "account_id", accounts, "account_id", "opportunities", "accounts")
orphan_keys(opportunity_contacts, "opportunity_id", opportunities, "opportunity_id",
            "opportunity_contacts", "opportunities")
orphan_keys(opportunity_contacts, "contact_id", contacts, "contact_id",
            "opportunity_contacts", "contacts")


In [ ]:
print(opportunities["stage"].value_counts(dropna=False)) 

### Hallazgos — `crm`

- Nulos:
- Duplicados:
- Llaves huérfanas:
- Inconsistencias categóricas (mayúsculas/minúsculas, variantes de texto):


---
## Auditoría de tipos y formatos (fechas, montos, texto)



In [ ]:
import re

def format_signature(series, max_samples=2000):
    """Convierte cada valor a una 'firma' de formato (dígitos->D, letras->A),
    para detectar mezclas de formato -- por ejemplo fechas en distintos patrones
    dentro de la misma columna."""
    def sig(v):
        s = str(v)
        s = re.sub(r'[A-Za-z]', 'A', s)
        s = re.sub(r'\d', 'D', s)
        return s
    sample = series.dropna()
    sample = sample[sample != ""]
    if len(sample) > max_samples:
        sample = sample.sample(max_samples, random_state=42)
    return sample.map(sig).value_counts()


def audit_column_types(df, table_name):
    """Revisa cada columna de la tabla y reporta inconsistencias de formato
    según lo que su nombre sugiere que debería contener."""
    print(f"\n=== {table_name} ===")
    found_issue = False
    for col in df.columns:
        if col.startswith("_source") or col == "_ingested_at":
            continue
        s = df[col]
        non_null = s[(s.notna()) & (s != "")]
        if len(non_null) == 0:
            continue
        lc = col.lower()

        # Columnas de fecha
        if "date" in lc or lc.endswith("_at") or lc.endswith("_on"):
            parsed = pd.to_datetime(non_null, errors="coerce")
            n_fail = parsed.isna().sum()
            sig = format_signature(non_null)
            if n_fail > 0 or len(sig) > 1:
                found_issue = True
                print(f"  [FECHA] {col}: {n_fail} valores no parseables | {len(sig)} formato(s) distinto(s) detectado(s)")
                print(sig.head(5).to_string())

        # Columnas numéricas / montos
        elif any(k in lc for k in ["amount", "price", "total", "rate", "grade",
                                     "score", "balance", "qty", "quantity", "cost"]):
            cleaned = non_null.astype(str).str.replace(r'[,$\s]', '', regex=True)
            numeric = pd.to_numeric(cleaned, errors="coerce")
            n_fail = numeric.isna().sum()
            if n_fail > 0:
                found_issue = True
                print(f"  [NUMÉRICO] {col}: {n_fail} valores no numéricos. Ejemplos: "
                      f"{non_null[numeric.isna()].unique()[:5].tolist()}")
            negatives = (numeric < 0).sum()
            if negatives > 0:
                found_issue = True
                print(f"  [NUMÉRICO] {col}: {negatives} valores negativos (revisar si es válido para el negocio)")

        # Texto / categórico
        elif not lc.endswith("_id") and lc != "id":
            non_null_str = non_null.astype(str)
            has_ws = (non_null_str.str.strip() != non_null_str).sum()
            n_unique = non_null_str.nunique()
            n_unique_lower = non_null_str.str.lower().str.strip().nunique()
            if has_ws > 0:
                found_issue = True
                print(f"  [TEXTO] {col}: {has_ws} valores con espacios extra al inicio/final")
            if n_unique_lower < n_unique and n_unique < 100:
                found_issue = True
                print(f"  [TEXTO] {col}: {n_unique} valores únicos, pero solo {n_unique_lower} "
                      f"al normalizar mayúsculas/espacios -> hay variantes del mismo valor")
    if not found_issue:
        print("  Sin inconsistencias de formato detectadas.")


### Aplicar la auditoría a las tablas de `university`

In [ ]:
for name, df in [("semesters", semesters), ("professors", professors), ("students", students),
                  ("courses", courses), ("enrollments", enrollments), ("grades", grades)]:
    audit_column_types(df, name)


### Aplicar la auditoría a las tablas de `billing`

In [ ]:
for name, df in [("customers", customers), ("products", products), ("subscriptions", subscriptions),
                  ("invoices", invoices), ("invoice_items", invoice_items), ("payments", payments)]:
    audit_column_types(df, name)


### Aplicar la auditoría a las tablas de `crm`

In [ ]:
for name, df in [("accounts", accounts), ("contacts", contacts), ("leads", leads),
                  ("opportunities", opportunities), ("opportunity_contacts", opportunity_contacts),
                  ("activities", activities)]:
    audit_column_types(df, name)


### Hallazgos — Auditoría de tipos y formatos

- Fechas con formatos mixtos:
- Montos/números no convertibles:
- Valores negativos donde no deberían existir:
- Texto con espacios extra o variantes de mayúsculas/minúsculas:

---
## Validación de reglas de negocio


In [ ]:
def check_rule(df, condition_mask, table_name, rule_description, id_cols):
    """Aplica una regla de negocio (boolean mask) y reporta cuántas filas la violan.
    condition_mask debe ser True donde la fila ES VÁLIDA; se reportan las que NO cumplen."""
    invalid = df[~condition_mask]
    pct = len(invalid) / len(df) * 100 if len(df) else 0
    print(f"[{table_name}] {rule_description}")
    print(f"  -> {len(invalid)} de {len(df)} filas violan la regla ({pct:.2f}%)")
    if len(invalid) > 0:
        display(invalid[id_cols].head(5))
    return invalid


### `crm.opportunities` — cierre no puede ser una fecha pasada a la creación

In [ ]:
opportunities["created_at_p"] = pd.to_datetime(opportunities["created_at"])
opportunities["close_date_p"] = pd.to_datetime(opportunities["close_date"])

mask = opportunities["close_date_p"] >= opportunities["created_at_p"]
invalid_opps = check_rule(
    opportunities, mask, "opportunities",
    "close_date debe ser >= created_at",
    ["opportunity_id", "created_at", "close_date"]
)


### `crm.activities` — actividad no puede ocurrir antes de crear la oportunidad relacionada

In [ ]:
activities_merged = activities.merge(
    opportunities[["opportunity_id", "created_at"]],
    on="opportunity_id", how="left"
)
activities_merged["occurred_at_p"] = pd.to_datetime(activities_merged["occurred_at"])

# Solo evaluamos filas donde sí hay opportunity_id (recordar: hay nulos ahí, ya detectados antes)
has_opp = activities_merged["created_at"].notna()
mask = ~has_opp | (activities_merged["occurred_at_p"] >= activities_merged["created_at"])
invalid_act = check_rule(
    activities_merged, mask, "activities",
    "occurred_at debe ser >= created_at de la oportunidad relacionada",
    ["activity_id", "occurred_at", "opportunity_id", "created_at"]
)


### `billing.invoices` — vencimiento no puede ser antes de la emisión

In [ ]:
invoices["issued_at_p"] = pd.to_datetime(invoices["issued_at"])
invoices["due_at_p"] = pd.to_datetime(invoices["due_at"])

mask = invoices["due_at_p"] >= invoices["issued_at_p"]
invalid_inv = check_rule(
    invoices, mask, "invoices",
    "due_at debe ser >= issued_at",
    ["invoice_id", "issued_at", "due_at"]
)


### `billing.payments` — el pago no puede ocurrir antes de emitir la factura

In [ ]:
payments_merged = payments.merge(
    invoices[["invoice_id", "issued_at_p"]],
    on="invoice_id", how="left"
)
payments_merged["paid_at_p"] = pd.to_datetime(payments_merged["paid_at"])

mask = payments_merged["paid_at_p"] >= payments_merged["issued_at_p"]
invalid_pay = check_rule(
    payments_merged, mask, "payments",
    "paid_at debe ser >= issued_at de la factura relacionada",
    ["payment_id", "paid_at", "invoice_id", "issued_at_p"]
)


### `billing.subscriptions` — la suscripción no puede terminar antes de empezar

In [ ]:
subscriptions["start_date_p"] = pd.to_datetime(subscriptions["start_date"])
subscriptions["end_date_p"] = pd.to_datetime(subscriptions["end_date"], errors="coerce")

# end_date puede ser nulo (suscripción activa/sin terminar) -- eso NO es un error
has_end = subscriptions["end_date_p"].notna()
mask = ~has_end | (subscriptions["end_date_p"] >= subscriptions["start_date_p"])
invalid_subs = check_rule(
    subscriptions, mask, "subscriptions",
    "end_date debe ser >= start_date (cuando end_date existe)",
    ["subscription_id", "start_date", "end_date"]
)


### `university.students` — edad razonable al momento de inscribirse

In [ ]:
students["birth_date_p"] = pd.to_datetime(students["birth_date"])
students["enrolled_at_p"] = pd.to_datetime(students["enrolled_at"])

age_at_enroll = (students["enrolled_at_p"] - students["birth_date_p"]).dt.days / 365.25

# Rango razonable para un estudiante: 15 a 90 años. Ajusta si el negocio real difiere.
mask = age_at_enroll.between(15, 90)
students["_age_at_enroll"] = age_at_enroll
invalid_students = check_rule(
    students, mask, "students",
    "edad al inscribirse debe estar entre 15 y 90 años",
    ["student_id", "birth_date", "enrolled_at", "_age_at_enroll"]
)


### `university.grades` — la nota no puede registrarse antes de la inscripción

In [ ]:
grades_merged = grades.merge(
    enrollments[["enrollment_id", "enrolled_at"]].rename(columns={"enrolled_at": "enr_enrolled_at"}),
    on="enrollment_id", how="left"
)
grades_merged["graded_at_p"] = pd.to_datetime(grades_merged["graded_at"])
grades_merged["enr_enrolled_at_p"] = pd.to_datetime(grades_merged["enr_enrolled_at"])

mask = grades_merged["graded_at_p"] >= grades_merged["enr_enrolled_at_p"]
invalid_grades = check_rule(
    grades_merged, mask, "grades",
    "graded_at debe ser >= enrolled_at de la inscripción relacionada",
    ["grade_id", "graded_at", "enrollment_id", "enr_enrolled_at"]
)


### Hallazgos — Reglas de negocio

| Regla | Tabla | Filas afectadas | % | Decisión propuesta para Silver |
|---|---|---|---|---|
| close_date >= created_at | opportunities | 34.30% | | |
| occurred_at >= created_at (oportunidad) | activities | 18.96% | | |
| due_at >= issued_at | invoices | 0.00% | | |
| paid_at >= issued_at | payments | 0.00% | | |
| end_date >= start_date | subscriptions | 5.22% | | |
| edad 15-90 al inscribirse | students | 12.72% |  | |
| graded_at >= enrolled_at | grades | 48.73% | | |


In [ ]:
students = load_table("university_students")
customers = load_table("billing_customers")

common_emails = set(students["email"]) & set(customers["email"])
print(f"Estudiantes que también son clientes de billing: {len(common_emails)} de {len(students)}")

In [ ]:
contacts = load_table("crm_contacts")

common_students_contacts = set(students["email"]) & set(contacts["email"])
common_customers_contacts = set(customers["email"]) & set(contacts["email"])

print(f"Estudiantes que también son contactos CRM: {len(common_students_contacts)} de {len(students)}")
print(f"Clientes billing que también son contactos CRM: {len(common_customers_contacts)} de {len(customers)}")

In [ ]:
pct_null = customers["external_ref"].isna().sum() / len(customers) * 100
print(f"external_ref: {pct_null:.1f}% nulo")
print(customers["external_ref"].dropna().head(10).tolist())  # para ver qué formato tiene cuando SÍ existe